# 03 · Plant the data issues

Notebooks 01 and 02 built a **perfect** world. Real client data is never perfect, and spotting problems *before* modelling is a big part of the marketing-science job. This notebook takes the clean data and breaks it in realistic ways, **only in `observed.duckdb`**.

```
truth.duckdb (clean)                      observed.duckdb (what the client sends)
  media_daily_clean    ──► break it ──►     media_daily
  media_national_clean ──► break it ──►     media_national_daily
  sales_daily_clean    ──► break it ──►     sales_daily
                                            
  issue_register  ◄── every issue logged here (the answer key for your data-quality checks)
```

| # | Issue | Table hit | Real-world cause | What it breaks if missed |
|---|---|---|---|---|
| 1 | TikTok tracking gap (2 weeks) | media_daily, media_national_daily | API export failed, rows never arrived | MMM reads "TikTok spend = 0" while TikTok was still driving sales |
| 2 | Duplicate load (3 days) | media_daily | export loaded twice | spend double-counted on those days |
| 3 | Negative spend | media_daily | platform credit booked as negative spend | impossible values, breaks log/saturation transforms |
| 4 | Region code rename | sales_daily | CRM migration gave Manchester a new code | Manchester "loses" all sales from April 2025; a new unknown region appears |
| 5 | Late-arriving sales | sales_daily | data pulled before the last days fully landed | last days look like a sales crash |

Two more traps live elsewhere and are logged in the register too:
- **Meta ↔ TikTok collinearity** (Feb–May 2025) was planted in 01, in the *true* media plan. It isn't a data error, so it isn't "fixed" by cleaning. It's a design problem the MMM has to live with.
- **Meta attribution window change** needs platform-attributed conversions, which we build in the attribution notebook. It will be planted there.

**Re-runnable:** this notebook always starts from the clean tables in truth, so running it twice gives the same result. Run order: `01 → 02 → 03`.

All issue settings (dates, regions, channels) live in `config/world.yaml` under `data_issues`.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, yaml, duckdb

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = yaml.safe_load((ROOT / "config/world.yaml").read_text())
iss = cfg["data_issues"]
rng = np.random.default_rng([cfg["seed"], 3])     # own random stream for this step

# Always start from the CLEAN tables (this is what makes the notebook re-runnable)
con = duckdb.connect(str(ROOT / "truth/truth.duckdb"), read_only=True)
media    = con.sql("select date, region_id, channel, spend, impressions, clicks from media_daily_clean").df()
national = con.sql("select * from media_national_clean").df()
sales    = con.sql("select * from sales_daily_clean").df()
con.close()

register = []          # one dict per planted issue -> truth.issue_register
def log_issue(issue, table, rows_affected, description, should_be_caught_by, impact_if_missed,
              channel=None, region_id=None, start=None, end=None, planted_in="03_plant_issues"):
    register.append(dict(issue_id=len(register) + 1, issue=issue, table_name=table, channel=channel,
                         region_id=region_id, start_date=pd.Timestamp(start) if start else pd.NaT,
                         end_date=pd.Timestamp(end) if end else pd.NaT, rows_affected=rows_affected,
                         planted_in=planted_in, description=description,
                         should_be_caught_by=should_be_caught_by, impact_if_missed=impact_if_missed))

print(f"clean rows  media_daily: {len(media):,}  media_national: {len(national):,}  sales_daily: {len(sales):,}")

## 1. TikTok tracking gap

TikTok's reporting export broke for two weeks, so **those rows are simply missing**, in both the regional and the national table. TikTok ads were still running in the real world (truth still has them), so TikTok was still driving sales.

**Why it matters:** if you `fillna(0)` or just don't notice, the MMM sees two weeks of zero TikTok spend while sales barely move, and concludes TikTok does less than it does.

**How you should catch it:** row completeness check: every `date × region × channel` should have exactly one row.

In [ ]:
g = iss["tiktok_tracking_gap"]
def in_gap(df):
    return (df["channel"] == g["channel"]) & df["date"].between(g["start"], g["end"])

n_regional, n_national = in_gap(media).sum(), in_gap(national).sum()
media    = media[~in_gap(media)].reset_index(drop=True)
national = national[~in_gap(national)].reset_index(drop=True)

log_issue("tracking_gap", "media_daily, media_national_daily", int(n_regional + n_national),
          f"{g['channel']} rows missing {g['start']} to {g['end']} (export failure)",
          "row completeness per date x region x channel",
          "MMM under-credits TikTok if missing rows are read as zero spend",
          channel=g["channel"], start=g["start"], end=g["end"])
print(f"dropped {n_regional:,} regional + {n_national} national TikTok rows")

## 2. Duplicate load

The ad-platform export for three days was loaded **twice** into `media_daily`. Every row on those days now appears two times. The national table was loaded separately and is fine, so regional totals no longer match national totals on those days. That mismatch is a second way to spot it.

**Why it matters:** spend (and impressions) look doubled on those days, which creates a fake spike.

**How you should catch it:** duplicate-key check: `group by date, region_id, channel having count(*) > 1`.

In [ ]:
d = iss["duplicate_load"]
dup_days = pd.to_datetime(d["dates"])
dupes = media[media["date"].isin(dup_days)]
media = pd.concat([media, dupes], ignore_index=True).sort_values(["channel", "date", "region_id"]).reset_index(drop=True)

log_issue("duplicate_rows", "media_daily", len(dupes),
          f"all media_daily rows on {', '.join(d['dates'])} loaded twice",
          "duplicate key check; regional sum vs national total",
          "spend double-counted on those days -> fake spike in MMM input",
          start=d["dates"][0], end=d["dates"][-1])
print(f"added {len(dupes):,} duplicate rows")

## 3. Negative spend

On one day, a platform credit (refund for bad ad delivery) was booked as **negative spend** for a few regions. Impressions and clicks are unchanged, so you get negative spend next to normal delivery, which is impossible.

**Why it matters:** negative inputs break saturation and log transforms, and they make cost per click / CPM nonsense.

**How you should catch it:** range check: `spend < 0`, `impressions < 0`, `clicks > impressions`.

In [ ]:
n = iss["negative_spend"]
day_rows = media.index[(media["channel"] == n["channel"]) & (media["date"] == n["date"])]
hit = rng.choice(day_rows, size=n["n_regions"], replace=False)
media.loc[hit, "spend"] = -(media.loc[hit, "spend"] * 0.5).round(2)    # credit worth half the day's spend
hit_regions = sorted(media.loc[hit, "region_id"])

log_issue("negative_values", "media_daily", len(hit),
          f"{n['channel']} spend negative on {n['date']} in {', '.join(hit_regions)} (credit booked as spend)",
          "range check: spend < 0",
          "breaks saturation / log transforms; CPM and CPC become negative",
          channel=n["channel"], region_id=", ".join(hit_regions), start=n["date"], end=n["date"])
media.loc[hit, ["date", "region_id", "channel", "spend", "impressions"]]

## 4. Region code rename

The client moved to a new CRM in April 2025, and Manchester's code changed from `R06` to `MCR01`, **but only in the sales system**. The `regions` table and the ad platforms still use `R06`.

Result: from April 2025, `R06` has no sales, and a code appears that isn't in `regions`.

**Why it matters:** a naive join drops Manchester's sales from April 2025 (national sales dip), or Manchester looks like it collapsed. In the geo test it could be picked as a "control" with fake data.

**How you should catch it:** referential integrity (sales `region_id` not in `regions`) and completeness per region over time.

In [ ]:
r = iss["region_code_rename"]
mask = (sales["region_id"] == r["old_code"]) & (sales["date"] >= r["from"])
sales.loc[mask, "region_id"] = r["new_code"]

log_issue("region_code_change", "sales_daily", int(mask.sum()),
          f"{r['old_code']} relabelled {r['new_code']} in sales from {r['from']} (CRM migration)",
          "orphan keys (sales region_id not in regions); per-region completeness over time",
          "Manchester sales vanish after the switch; joins silently drop revenue",
          region_id=f"{r['old_code']} -> {r['new_code']}", start=r["from"], end=cfg["dates"]["end"])
print(f"relabelled {mask.sum()} rows")

## 5. Late-arriving sales

The extract was pulled before the last few days had fully landed (returns processing, slow store feeds). The final days are **incomplete**: 85%, 60% and 30% of their true value.

**Why it matters:** the last days look like a crash in sales, at peak season. A model may read it as "the end-of-December ads didn't work", or forecast a slump.

**How you should catch it:** compare the last days to the same weekday in previous weeks / last year; a step change at the very end of a series is a classic "data not complete yet" signature.

In [ ]:
l = iss["late_arriving_sales"]
n_rows = 0
for day, frac in l["completeness"].items():
    m = sales["date"] == pd.Timestamp(day)
    sales.loc[m, "orders"]  = np.round(sales.loc[m, "orders"] * frac).astype("int64")
    sales.loc[m, "revenue"] = (sales.loc[m, "revenue"] * frac).round(2)
    n_rows += int(m.sum())

days = list(l["completeness"])
log_issue("incomplete_recent_data", "sales_daily", n_rows,
          "last days only partly loaded: " + ", ".join(f"{d} {int(f*100)}%" for d, f in l["completeness"].items()),
          "tail check: last days vs same weekday in prior weeks",
          "fake sales drop at year end; biases trend and late-December media effects",
          start=days[0], end=days[-1])
print(f"scaled {n_rows} rows")

## 6. Traps planted elsewhere (logged for completeness)

These go in the register so it is the **single list** of everything that should trip up an analyst, but this notebook doesn't change any data for them.

In [ ]:
cw = cfg["budget_planning"]["collinear_window"]
log_issue("collinearity", "media_daily (true plan)", 0,
          f"{cw['follower']} budget copies {cw['leader']} shape {cw['start']} to {cw['end']}",
          "correlation / VIF between channel spends by period",
          "MMM can't separate Meta from TikTok in that window -> wide or swapped estimates",
          channel=f"{cw['leader']}, {cw['follower']}", start=cw["start"], end=cw["end"],
          planted_in="01_build_world")
log_issue("attribution_window_change", "platform attribution (to be built)", 0,
          "Meta switches 7-day click to 1-day click attribution (planned)",
          "step change in platform-reported conversions with no change in spend",
          "platform ROAS drops overnight with no real change in performance",
          channel="meta", planted_in="planned: attribution notebook")
issue_register = pd.DataFrame(register)
issue_register[["issue_id", "issue", "table_name", "channel", "region_id", "start_date", "end_date", "rows_affected", "planted_in"]]

## 7. Write

Overwrite the three observed tables with the broken versions and save the register to truth. `regions`, `calendar` and `promotions` are left as they are.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from src.simulate.store import write_tables

write_tables(ROOT / "data/observed.duckdb", {
    "media_daily": media,
    "media_national_daily": national,
    "sales_daily": sales,
})
write_tables(ROOT / "truth/truth.duckdb", {"issue_register": issue_register})
print("written")

## 8. Did it work? Observed vs clean

A quick diff to confirm each issue landed. These are checks on the *simulator*; the real test comes in the data-quality notebook, where you'll try to find these **without** looking at truth.

In [ ]:
con = duckdb.connect(str(ROOT / "data/observed.duckdb"), read_only=True)
con.execute(f"ATTACH '{ROOT / 'truth/truth.duckdb'}' AS t (READ_ONLY)")
q = lambda s: con.sql(s).df()

print("Row counts: observed vs clean")
display(q('''
    select 'media_daily' as tbl, (select count(*) from media_daily) as observed, (select count(*) from t.media_daily_clean) as clean
    union all select 'media_national_daily', (select count(*) from media_national_daily), (select count(*) from t.media_national_clean)
    union all select 'sales_daily', (select count(*) from sales_daily), (select count(*) from t.sales_daily_clean)'''))

print("1. TikTok rows per day around the gap")
display(q('''select date, count(*) as tiktok_rows from media_daily
             where channel = 'tiktok' and date between '2024-08-10' and '2024-08-27' group by 1 order by 1'''))

print("2. Duplicate keys")
display(q('''select date, count(*) as dup_keys from (
               select date, region_id, channel from media_daily group by all having count(*) > 1)
             group by 1 order by 1'''))

print("3. Negative spend")
display(q("select * from media_daily where spend < 0"))

print("4. Sales region codes not in regions table")
display(q('''select s.region_id, min(date) as first_seen, count(*) as n_rows from sales_daily s
             anti join regions r using (region_id) group by 1'''))

print("5. Last days: observed vs clean national revenue")
display(q('''select o.date, round(sum(o.revenue)) as observed, round(sum(c.revenue)) as clean,
                    round(sum(o.revenue) / sum(c.revenue), 2) as ratio
             from sales_daily o join t.sales_daily_clean c using (date, region_id)
             where o.date >= '2025-12-26' group by 1 order by 1'''))
con.close()